In [24]:
import pandas as pd
import numpy as np
import boto3
from sklearn.model_selection import train_test_split
import sagemaker
from sagemaker import Session
import io
import sagemaker.amazon.common as smac
import os
from sagemaker.amazon.amazon_estimator import get_image_uri

In [25]:
df = pd.read_csv("loan_dataset.csv")
df.head()

,Gender,Marital_Status,Education,Self_Employed,Applicant_Income,Coapplicant_Income,Loan_Amount_INR,Loan_Term_Years,Credit_History,Existing_Loans,Loan_Status
0,1,1,1,0,60154,57252,2241607,5,1,0,1
1,0,0,0,0,140466,74736,303525,10,1,2,1
2,1,0,0,1,43253,29851,1506744,10,0,0,0
3,1,0,0,1,95940,8159,2435311,30,1,0,1
4,1,0,0,1,185476,57475,920616,10,1,2,1


In [26]:
df.shape

(150000, 11)

In [27]:
X = df.drop(columns=["Loan_Status"])
y = df[["Loan_Status"]]

In [28]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [29]:
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

In [30]:
X_train = X_train.astype("float32")
y_train = y_train.astype("float32")

X_test  = X_test.astype("float32")
y_test  = y_test.astype("float32")

In [31]:
X_train

,Gender,Marital_Status,Education,Self_Employed,Applicant_Income,Coapplicant_Income,Loan_Amount_INR,Loan_Term_Years,Credit_History,Existing_Loans
0,1.0,0.0,1.0,1.0,82484.0,29647.0,910384.0,10.0,1.0,0.0
1,1.0,1.0,0.0,1.0,189045.0,8978.0,943417.0,30.0,1.0,2.0
2,0.0,1.0,0.0,0.0,167888.0,18658.0,1093965.0,25.0,0.0,0.0
3,1.0,1.0,1.0,1.0,64484.0,21575.0,2488758.0,25.0,0.0,2.0
4,0.0,1.0,0.0,0.0,172177.0,37688.0,1066039.0,5.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...
119995,0.0,1.0,1.0,1.0,79498.0,54303.0,247094.0,20.0,1.0,2.0
119996,1.0,1.0,1.0,0.0,18884.0,71057.0,915417.0,30.0,1.0,2.0
119997,1.0,0.0,1.0,1.0,49989.0,78812.0,436692.0,25.0,0.0,0.0
119998,0.0,1.0,1.0,0.0,61425.0,29923.0,1787295.0,20.0,1.0,0.0


In [32]:
X_test

,Gender,Marital_Status,Education,Self_Employed,Applicant_Income,Coapplicant_Income,Loan_Amount_INR,Loan_Term_Years,Credit_History,Existing_Loans
0,0.0,0.0,0.0,1.0,116855.0,61104.0,1683190.0,15.0,1.0,1.0
1,1.0,0.0,1.0,1.0,100744.0,64858.0,1699382.0,5.0,1.0,2.0
2,0.0,0.0,1.0,0.0,153522.0,62193.0,1193446.0,5.0,1.0,0.0
3,0.0,1.0,1.0,0.0,149489.0,79550.0,970798.0,15.0,1.0,1.0
4,1.0,1.0,0.0,1.0,101723.0,19255.0,2060509.0,10.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
29995,0.0,1.0,1.0,0.0,62390.0,69441.0,2365016.0,5.0,1.0,1.0
29996,0.0,0.0,1.0,1.0,21117.0,55635.0,2134204.0,25.0,1.0,2.0
29997,0.0,1.0,1.0,1.0,44043.0,58323.0,1133394.0,15.0,0.0,2.0
29998,1.0,1.0,1.0,1.0,194825.0,54434.0,771999.0,30.0,1.0,1.0


In [33]:
y_train = y_train.iloc[:,0]

In [34]:
y_train

0         1.0
1         1.0
2         1.0
3         0.0
4         1.0
         ... 
119995    1.0
119996    1.0
119997    1.0
119998    1.0
119999    1.0
Name: Loan_Status, Length: 120000, dtype: float32

In [35]:
y_test = y_test.iloc[:,0]

In [36]:
sagemaker_session = sagemaker.Session()

bucket_name = "loan-predict"

prefix="linear-predict"

role= sagemaker.get_execution_role()

print(role)

arn:aws:iam::118903272754:role/service-role/AmazonSageMaker-ExecutionRole-20260301T152362


In [37]:
X_train = np.array(X_train)

In [38]:
bucket_name

'loan-predict'

In [39]:
buf = io.BytesIO()

smac.write_numpy_to_dense_tensor(buf, X_train, y_train)
buf.seek(0)

0

In [40]:
key = "loan-data"

boto3.resource("s3").Bucket(bucket_name).Object(os.path.join(prefix, "train", key)).upload_fileobj(buf)

s3_train_data = f"s3://{bucket_name}/{prefix}/train/{key}"

print("Data Uploaded:", s3_train_data)

Data Uploaded: s3://loan-predict/linear-predict/train/loan-data


In [41]:
X_test = np.array(X_test)

buf = io.BytesIO()

smac.write_numpy_to_dense_tensor(buf,X_test,y_test)
buf.seek(0)

key = "loan-date-test"

boto3.resource("s3").Bucket(bucket_name).Object(os.path.join(prefix, "test", key)).upload_fileobj(buf)

s3_train_data = f"s3://{bucket_name}/{prefix}/test/{key}"

print("Data Uploaded:", s3_train_data)

Data Uploaded: s3://loan-predict/linear-predict/test/loan-date-test


In [42]:
output_loaction = f"s3://{bucket_name}/{prefix}/output"
output_loaction

's3://loan-predict/linear-predict/output'

In [43]:
container = sagemaker.image_uris.retrieve("linear-learner",boto3.Session().region_name)

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [44]:
linear = sagemaker.estimator.Estimator(container,role,instance_count=1,instance_type="ml.m5.large",output_path = output_loaction,sagemaker_session=sagemaker_session)

In [45]:
linear.set_hyperparameters(feature_dim=10,predictor_type="regressor",mini_batch_size=4,epochs=6,num_models=32,loss="absolute_loss")

In [46]:
linear.fit({"train":s3_train_data})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: linear-learner-2026-03-25-02-39-52-376


2026-03-25 02:39:52 Starting - Starting the training job...
2026-03-25 02:40:13 Starting - Preparing the instances for training...
2026-03-25 02:40:35 Downloading - Downloading input data...
2026-03-25 02:41:10 Downloading - Downloading the training image.........
2026-03-25 02:42:41 Training - Training image download completed. Training in progress.Docker entrypoint called with argument(s): train
Running default environment configuration script
[03/25/2026 02:42:43 INFO 139716644329280] Reading default configuration from /opt/amazon/lib/python3.8/site-packages/algorithm/resources/default-input.json: {'mini_batch_size': '1000', 'epochs': '15', 'feature_dim': 'auto', 'use_bias': 'true', 'binary_classifier_model_selection_criteria': 'accuracy', 'f_beta': '1.0', 'target_recall': '0.8', 'target_precision': '0.8', 'num_models': 'auto', 'num_calibration_samples': '10000000', 'init_method': 'uniform', 'init_scale': '0.07', 'init_sigma': '0.01', 'init_bias': '0.0', 'optimizer': 'auto', 'loss':

In [47]:
linear_regresor = linear.deploy(initial_instance_count=1,instance_type="ml.m5.large")

INFO:sagemaker:Creating model with name: linear-learner-2026-03-25-02-53-26-050
INFO:sagemaker:Creating endpoint-config with name linear-learner-2026-03-25-02-53-26-050
INFO:sagemaker:Creating endpoint with name linear-learner-2026-03-25-02-53-26-050


-------!

In [48]:
linear_regresor.serializer = sagemaker.serializers.CSVSerializer()
linear_regresor.deserializer = sagemaker.deserializers.JSONDeserializer()

In [49]:
result = linear_regresor.predict(X_test)

In [50]:
result

{'predictions': [{'score': 1.000004529953003},
  {'score': 0.9999977946281433},
  {'score': 0.9999905228614807},
  {'score': 0.9999881982803345},
  {'score': -2.4237935576820746e-05},
  {'score': -2.034776844084263e-05},
  {'score': 0.9999919533729553},
  {'score': 0.9999794960021973},
  {'score': 0.9999815225601196},
  {'score': 0.9999939799308777},
  {'score': 0.9999855160713196},
  {'score': 0.9999948143959045},
  {'score': 0.9999849200248718},
  {'score': 0.9999756217002869},
  {'score': 0.9999884366989136},
  {'score': 0.9999917149543762},
  {'score': 0.9999935030937195},
  {'score': 0.9999693632125854},
  {'score': 0.9999883770942688},
  {'score': 0.9999827146530151},
  {'score': 0.9999880194664001},
  {'score': -1.2662278095376678e-05},
  {'score': 0.9999834299087524},
  {'score': 0.9999733567237854},
  {'score': 0.9999752640724182},
  {'score': -2.5526705940137617e-05},
  {'score': -1.5947418432915583e-05},
  {'score': -1.087788950826507e-05},
  {'score': -2.5192452085320838e-0